# 3. Modelling: Forecasting Carbon Intensity (XGBoost & LSTM)

**Project:** AI for low‑carbon energy scheduling: forecasting electricity carbon intensity and recommending cleaner time windows

**Purpose:** This notebook trains the first two models (XGBoost and LSTM) to predict future Grid Carbon Intensity. These models provide the "intelligence" for our low-carbon scheduler.

**Course Reference:** 
- **XGBoost Implementation:** Follows tabular regression standards established in **Lab 02**.
- **LSTM Implementation:** Sequence processing and windowing techniques are adapted from **Lab 04 (DL4TS)**, focusing on recurrent architectures for sustainability forecasting.

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt
import os
import pickle

# --- Setup Directories ---
PROCESSED_DATA_DIR = 'data/processed/'
MODELS_DIR = 'models/'
if not os.path.exists(MODELS_DIR): os.makedirs(MODELS_DIR)

# Load Datasets (Time-series split from Notebook 1)
train_df = pd.read_csv(os.path.join(PROCESSED_DATA_DIR, 'train_ci.csv'), parse_dates=['timestamp']).set_index('timestamp')
val_df = pd.read_csv(os.path.join(PROCESSED_DATA_DIR, 'val_ci.csv'), parse_dates=['timestamp']).set_index('timestamp')
test_df = pd.read_csv(os.path.join(PROCESSED_DATA_DIR, 'test_ci.csv'), parse_dates=['timestamp']).set_index('timestamp')

# --- Define Features and Target ---
features = ['hour', 'day_of_week', 'intensity_lag_30m', 'intensity_lag_1h', 'intensity_lag_24h', 'intensity_rolling_mean_6h', 'demand_lag_30m']
target = 'carbon_intensity'

## 3.1 Data Scaling

**Methodology:** LSTMs are sensitive to feature scales. We use `StandardScaler` to ensure all inputs have a mean of 0 and standard deviation of 1.

**Ref:** Normalization strategy adapted from the preprocessing steps in **Lab 04**.

In [ ]:
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_train = scaler_X.fit_transform(train_df[features])
y_train = scaler_y.fit_transform(train_df[[target]]).flatten()

X_val = scaler_X.transform(val_df[features])
y_val = scaler_y.transform(val_df[[target]]).flatten()

X_test = scaler_X.transform(test_df[features])
y_test = scaler_y.transform(test_df[[target]]).flatten()

with open(os.path.join(MODELS_DIR, 'scalers_ci.pkl'), 'wb') as f:
    pickle.dump({'X': scaler_X, 'y': scaler_y}, f)

## 3.2 Model 1: XGBoost Regressor

**Methodology:** XGBoost provides an extremely efficient tabular baseline. We use early stopping on the validation set to prevent overfitting.

**Source:** Hyperparameter choices adapted from official XGBoost documentation for time-series forecasting.

In [ ]:
xgb_model = xgb.XGBRegressor(
    n_estimators=500,        # Maximum number of trees
    learning_rate=0.05,     # Weight reduction for each step
    max_depth=6,            # Limit tree depth to control complexity
    subsample=0.8,          # Use a subset of data per tree for robustness
    colsample_bytree=0.8,   # Feature sampling
    n_jobs=-1,              # Parallel processing
    base_score=0.5,         # Default starting prediction
    early_stopping_rounds=20 # Stop if validation loss doesn't improve
)

xgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
xgb_model.save_model(os.path.join(MODELS_DIR, 'xgb_ci_model.json'))

## 3.3 Model 2: Long Short-Term Memory (LSTM)

**Methodology:** We reshape the data into 3D windows (Samples, Time Steps, Features) to allow the LSTM to capture temporal dependencies over a 6-hour look-back period (12 steps).

**Ref:** Windowing function and LSTM architecture (Dense-LSTM-Dense) adapted directly from **Lab 04 - Week 5**.

In [ ]:
def create_windows(X, y, window_size=12):
    """Converts 2D tabular data to 3D temporal windows for RNN ingestion."""
    X_win, y_win = [], []
    for i in range(len(X) - window_size):
        X_win.append(X[i:i+window_size])
        y_win.append(y[i+window_size])
    return np.array(X_win), np.array(y_win)

window_size = 12
X_train_win, y_train_win = create_windows(X_train, y_train, window_size)
X_val_win, y_val_win = create_windows(X_val, y_val, window_size)
X_test_win, y_test_win = create_windows(X_test, y_test, window_size)

lstm_model = Sequential([
    LSTM(64, input_shape=(window_size, len(features)), return_sequences=True),
    Dropout(0.2),  # Dropout used for regularization (Lab 04 technique)
    LSTM(32),
    Dense(1)       # Final regression output
])

lstm_model.compile(optimizer='adam', loss='mse')
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

lstm_model.fit(X_train_win, y_train_win, validation_data=(X_val_win, y_val_win), epochs=50, batch_size=32, callbacks=[early_stop], verbose=1)
lstm_model.save(os.path.join(MODELS_DIR, 'lstm_ci_model.h5'))

## 3.4 Evaluation Metrics

**Methodology:** We calculate MAE, RMSE, and R2 on the held-out test set to ensure the accuracy results are honest and representative of real-world performance.

**Ref:** Metrics selection based on **Lab 01 & 02** regression evaluation standards.

In [ ]:
def evaluate(model, X, y_true, scaler_y, name, is_lstm=False):
    y_pred = model.predict(X)
    if is_lstm: y_pred = y_pred.flatten()
    
    # Inverse transform to return to original gCO2/kWh scale
    y_pred_inv = scaler_y.inverse_transform(y_pred.reshape(-1, 1)).flatten()
    y_true_inv = scaler_y.inverse_transform(y_true.reshape(-1, 1)).flatten()
    
    print(f"--- {name} Final Test Results ---")
    print(f"MAE: {mean_absolute_error(y_true_inv, y_pred_inv):.4f} gCO2/kWh")
    print(f"RMSE: {np.sqrt(mean_squared_error(y_true_inv, y_pred_inv)):.4f} gCO2/kWh")
    print(f"R2 Score: {r2_score(y_true_inv, y_pred_inv):.4f}\n")

evaluate(xgb_model, X_test, y_test, scaler_y, "XGBoost")
evaluate(lstm_model, X_test_win, y_test_win, scaler_y, "LSTM", is_lstm=True)